# 🎬 CloudStream WebDAV Bridge (Zero-Cost 10 Gbps Cloud Runner)
Bypasses your 50 Mbps home internet bottleneck to stream 50 GB - 100 GB 4K UHD Remuxes.
- **Datacenter Speed**: Google Cloud 10 Gbps pipe pulls the 100 GB link at gigabits/sec.
- **On-Demand Range Proxy**: Sends only the exact 10 MB - 20 MB block you are watching to CX File Explorer or VLC.
- **Zero Storage**: 0 bytes used on your phone, TV, or PC.

In [ ]:
# 1. Install dependencies
!pip install -q fastapi "uvicorn[standard]" httpx pydantic aiofiles
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

In [ ]:
# 2. Launch CloudStream Bridge in background & establish public HTTPS tunnel
import subprocess, time, re

!nohup uvicorn main:app --host 0.0.0.0 --port 7860 > server.log 2>&1 &
time.sleep(2)

# Start Cloudflare tunnel
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:7860"], stderr=subprocess.PIPE, text=True)

print("[*] Establishing high-speed 10 Gbps public tunnel...")
for line in tunnel_proc.stderr:
    if "trycloudflare.com" in line:
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            print("\n" + "="*60)
            print(f"🎬 Web UI:     {public_url}")
            print(f"📁 WebDAV URL: {public_url}/dav/")
            print("="*60)
            break